#  Proyecto: Mantenimiento Predictivo Industrial
**Caso de Negocio:** Minimizar las paradas no programadas de maquinaria mediante la detección temprana de fallas operativas.
**Objetivo:** Desarrollar un modelo en Python para anticipar fallas críticas, optimizar el mantenimiento y reducir costos.


In [33]:
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler

print("=== 1. GENERANDO DATOS INDUSTRIALES SIMULADOS ===")
X_raw, y_raw = make_classification(
    n_samples=10000,
    n_features=5,
    n_informative=4,
    n_redundant=0,
    n_classes=2,
    weights=[0.99, 0.01],
    random_state=42
)

# Creamos el DataFrame con nombres de sensores reales
df = pd.DataFrame(X_raw, columns=['Temperatura', 'RPM', 'Torque', 'Vibracion', 'Desgaste'])
df['failure'] = y_raw
print(f"Registros totales: {len(df)} | Fallas reales: {df['failure'].sum()}")

print("\n=== 2. PROCESAMIENTO Y ESCALADO ===")
X = df.drop(columns=['failure'])
y = df['failure']

scaler = StandardScaler()
X_escalado = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_escalado, y, test_size=0.2, random_state=42)

print("\n=== 3. ENTRENAMIENTO DEL MODELO (RANDOM FOREST OPTIMIZADO) ===")
# Penalizamos 10 veces más el error en las fallas para maximizar el recall
modelo_ajustado = RandomForestClassifier(class_weight={0: 1, 1: 10}, random_state=42)
modelo_ajustado.fit(X_train, y_train)

predicciones = modelo_ajustado.predict(X_test)
print(classification_report(y_test, predicciones))

print("\n=== 4. INTERPRETABILIDAD: IMPORTANCIA DE SENSORES ===")
importances = modelo_ajustado.feature_importances_
df_importancia = pd.DataFrame({
    'Sensor / Variable': X.columns,
    'Importancia': importances
}).sort_values(by='Importancia', ascending=False)
display(df_importancia)

print("\n=== 5. ANÁLISIS DE RIESGO EN TIEMPO REAL ===")
probabilidades = modelo_ajustado.predict_proba(X_test)
df_riesgo = pd.DataFrame({
    'Probabilidad Normal (%)': probabilidades[:, 0] * 100,
    'Probabilidad de Falla (%)': probabilidades[:, 1] * 100,
    'Estado Real': y_test.values
}, index=y_test.index)

maquinas_en_peligro = df_riesgo[df_riesgo['Probabilidad de Falla (%)'] > 50]
print(f"Alertas críticas detectadas con >50% de riesgo: {len(maquinas_en_peligro)}")
display(maquinas_en_peligro.head())

=== 1. GENERANDO DATOS INDUSTRIALES SIMULADOS ===
Registros totales: 10000 | Fallas reales: 161

=== 2. PROCESAMIENTO Y ESCALADO ===

=== 3. ENTRENAMIENTO DEL MODELO (RANDOM FOREST OPTIMIZADO) ===
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      1968
           1       1.00      0.19      0.32        32

    accuracy                           0.99      2000
   macro avg       0.99      0.59      0.65      2000
weighted avg       0.99      0.99      0.98      2000


=== 4. INTERPRETABILIDAD: IMPORTANCIA DE SENSORES ===


,Sensor / Variable,Importancia
1,RPM,0.282083
3,Vibracion,0.233453
0,Temperatura,0.190607
2,Torque,0.168123
4,Desgaste,0.125734



=== 5. ANÁLISIS DE RIESGO EN TIEMPO REAL ===
Alertas críticas detectadas con >50% de riesgo: 6


,Probabilidad Normal (%),Probabilidad de Falla (%),Estado Real
4773,14.0,86.0,1
7590,41.0,59.0,1
9476,20.0,80.0,1
7566,14.0,86.0,1
6848,24.0,76.0,1
